# Phase 6 - Step 1: Abstractive Summarization data exploration

**Task:** read a legal document, write a short summary of it **in new words** (abstractive = the model generates
text, it does not only pick sentences). Metric: **BERTScore F1** (compares the meaning of the predicted and the
reference summary word by word with roberta-large embeddings); secondary ROUGE-1 F1 and ROUGE-L F1 (word overlap).

What this notebook finds out before we design any model (no training, GPU only for the BERTScore baselines):

1. download `Phase_6_Summ/` from Hugging Face (skipped if already on disk)
2. real file format + data checks (duplicates, train/test overlap, empty texts, hidden test summaries)
3. writing style (cased / lower-case, line breaks, placeholders) of documents **and** summaries
4. lengths in words: documents, summaries, **compression ratio**, how summary length depends on document length
5. lengths in **bpe32k tokens** -> what fits in my Phase-1 context of 1536 tokens (document + summary!)
6. **what the metric actually sees**: roberta-large reads at most 512 tokens -> are long summaries cut?
7. **how abstractive** are the summaries? (new words / n-grams, copied fragments) -> is the copy mechanism useful?
8. **where in the document** the summary content comes from (start / middle / end) and whether summaries follow the document order
9. a fixed **validation split** (there is no public dev set in Phase 6)
10. **baselines with the real metric** on the local-eval set: lead-k words for several k, tail, oracle extractive,
    "the same summary for every case" -> the floor, and how BERTScore / ROUGE change with summary length

Results are cached in `experiments/phase6/p6_01_data/` (re-running loads them).


In [1]:
import json, re, sys, collections
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# project root = first folder upwards that contains .gitignore (same as all earlier notebooks)
ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import summ_utils as su

RAW_DIR = ROOT / "data" / "raw"
P6 = su.summ_dir(ROOT)
OUT = ROOT / "experiments" / "phase6" / "p6_01_data"
OUT.mkdir(parents=True, exist_ok=True)
SPLIT_PATH = ROOT / "data" / "processed" / "phase6" / "split_v1.json"
TOK_PATH = ROOT / "data" / "tokenizer" / "bpe32k.json"
N_SAMPLE = 500          # train documents used for the slower statistics (sections 7-8)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("project root:", ROOT)

def load_cache(name):
    p = OUT / name
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None

def save_cache(name, obj):
    (OUT / name).write_text(json.dumps(obj, indent=1), encoding="utf-8")

def pct_row(x):
    x = np.asarray(x, dtype=float)
    return {"mean": round(float(x.mean()), 1), "p10": int(np.percentile(x, 10)), "p25": int(np.percentile(x, 25)),
            "p50": int(np.percentile(x, 50)), "p75": int(np.percentile(x, 75)), "p90": int(np.percentile(x, 90)),
            "p99": int(np.percentile(x, 99)), "max": int(x.max())}

def spearman(a, b):
    return float(np.corrcoef(pd.Series(a).rank().values, pd.Series(b).rank().values)[0, 1])

project root: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project


c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Download the Phase 6 files (skipped if already on disk)

The doc (section 12.3): `Phase_6_Summ/Train/train.json` and `Phase_6_Summ/Test/test.json`; **no dev set**.
The repository is gated: you must be logged in (`hf auth login`, done in earlier phases).


In [2]:
FILES = su.find_files(ROOT)
if {"train", "test"} <= set(FILES):
    print("Phase 6 data already downloaded -> skipping download")
else:
    from huggingface_hub import snapshot_download
    snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset",
                      allow_patterns="Phase_6_Summ/*", local_dir=str(RAW_DIR))
    FILES = su.find_files(ROOT)
for p in sorted(P6.rglob("*")):
    if p.is_file():
        print(f"{str(p.relative_to(P6)):30s} {p.stat().st_size / 1e6:9.2f} MB")
print("files used:", {k: str(v.relative_to(P6)) for k, v in FILES.items()})
extra = [p for p in P6.rglob("*") if p.is_file() and p not in FILES.values()]
if extra:
    print("NOTE: files not used by this notebook:", [str(p.relative_to(P6)) for p in extra])

Phase 6 data already downloaded -> skipping download
Test\test.json                      2.73 MB
Train\train.json                  210.60 MB
files used: {'train': 'Train\\train.json', 'test': 'Test\\test.json'}


## 2. Load the files and look at the real format

The doc shows `{"id", "document", "summary"}`, but earlier phases taught us to check instead of trusting:
the file can be a **list** or a **dict** `{id: record}`, a field can be one string or a **list of sentences**
(then joined with a space), and the key names may differ (`text`, `judgment`, `headnote` ...).
`su.load_summ` handles all of these and reports which keys it used.
Phase 4's test.json secretly contained labels -> if test has summaries, they are **dropped on load** and never used.


In [3]:
DATA, INFO = su.load_summ(files=FILES)
for split, rec in INFO.items():
    print(f"===== {split}: {rec}")
TR, TE = DATA["train"], DATA["test"]
if INFO["test"].get("test_had_summaries"):
    print("\nWARNING: test.json contained summaries -> dropped now, never used")
d0 = TR[0]
print(f"\nfirst train id={d0['id']!r}")
print("document starts:", repr(d0["document"][:600]))
print("summary  starts:", repr((d0["summary"] or "")[:600]))

===== train: {'file': 'train.json', 'file_type': 'list', 'records': 7030, 'keys': ['id', 'document', 'summary'], 'document_key': 'document', 'summary_key': 'summary', 'document_type': 'str', 'summary_type': 'str'}
===== test: {'file': 'test.json', 'file_type': 'list', 'records': 100, 'keys': ['id', 'document'], 'document_key': 'document', 'summary_key': None, 'document_type': 'str', 'summary_type': None}

first train id='6712'
document starts: 'IVIL Appeal No.768 (NT) of 1977 etc. From the judgment Order dated 9.10.1975 of the Madhya Pradesh High Court in M.C.C. No.144 of 1966. V.A. Bobde, B.R.Aggarwala and U.A.Rana for the Appellant. S.V.Deshpande and S.K.Agnihotri for the Respondents. The Judgment of the Court was delivered by RANGANATH MISHRA, CJ. These are appeals by special leave 252 and are directed against the separate decisions of the Madhya Pradesh High Court in references under the Madhya Pradesh Sales Tax Act. Civil Appeal No.768/77 relates to the assessment period 1951 52, 

## 3. Data checks

- duplicate ids, empty / very short documents, empty summaries
- line breaks (structure), non-ASCII characters (Phase 1 trained on cleaned ASCII text), PDF signature stamps,
  placeholders like `<ENTITY>` (Phase 5 had masked names)
- **duplicate documents** inside train and between train and test (a test document seen in training inflates
  scores), the same document with **different summaries**, and the same summary for different documents


In [4]:
PLACEHOLDER_RE = re.compile(r"<[A-Z_]{2,20}>")
rows = []
for split, docs in DATA.items():
    for field in ("document", "summary"):
        texts = [d[field] for d in docs if d[field] is not None]
        if not texts:
            continue
        n_chars = sum(len(t) for t in texts)
        rows.append({
            "split/field": f"{split}/{field}", "items": len(texts),
            "dup_ids": len(docs) - len({d['id'] for d in docs}) if field == "document" else "",
            "empty": sum(not t.strip() for t in texts),
            "<100 words": sum(len(t.split()) < 100 for t in texts),
            "with line breaks %": round(100 * np.mean(["\n" in t for t in texts]), 1),
            "line breaks / item": round(float(np.mean([t.count("\n") for t in texts])), 1),
            "non-ASCII %": round(100 * sum(sum(ord(c) > 127 for c in t) for t in tqdm(texts, desc=f"{split}/{field}", leave=False)) / max(n_chars, 1), 3),
            "signature stamps": sum("signature not verified" in t.lower() for t in texts),
            "with placeholders %": round(100 * np.mean([bool(PLACEHOLDER_RE.search(t)) for t in texts]), 1),
        })
display(pd.DataFrame(rows).set_index("split/field"))
ph = collections.Counter(m for d in TR for t in (d["document"], d["summary"] or "") for m in PLACEHOLDER_RE.findall(t))
print("placeholders found (train docs + summaries):", ph.most_common(10) or "none")

# duplicates (case/whitespace-insensitive fingerprints)
by_doc = collections.defaultdict(list)
for split, docs in DATA.items():
    for d in docs:
        by_doc[su.text_key(d["document"])].append((split, d["id"], su.text_key(d["summary"]) if d["summary"] else None))
groups = [v for v in by_doc.values() if len(v) > 1]
spans = collections.Counter("&".join(sorted({s for s, _, _ in v})) for v in groups)
diff_sum = [v for v in groups if len({h for _, _, h in v if h}) > 1]
print(f"\nduplicate documents: {len(groups)} groups ({sum(len(v) - 1 for v in groups)} extra copies); splits: {dict(spans)}")
print(f"  duplicate documents with DIFFERENT summaries: {len(diff_sum)}")
sum_groups = collections.Counter(su.text_key(d["summary"]) for d in TR if d["summary"])
same_sum = {h: c for h, c in sum_groups.items() if c > 1}
print(f"identical summaries used for different train documents: {len(same_sum)} groups ({sum(same_sum.values())} docs)")
print("ids shared between train and test:", len({d['id'] for d in TR} & {d['id'] for d in TE}))
DUP_INFO = {"dup_doc_groups": len(groups), "extra_copies": sum(len(v) - 1 for v in groups), "spans": dict(spans),
            "dup_docs_different_summaries": len(diff_sum), "identical_summary_groups": len(same_sum)}

,items,dup_ids,empty,<100 words,with line breaks %,line breaks / item,non-ASCII %,signature stamps,with placeholders %
split/field,,,,,,,,,
train/document,7030,0,0,1,0.0,0.0,0.0,0,0.0
train/summary,7030,,2,37,0.0,0.0,0.0,0,0.0
test/document,100,0,0,0,0.0,0.0,0.0,0,0.0


placeholders found (train docs + summaries): none

duplicate documents: 27 groups (27 extra copies); splits: {'train': 27}
  duplicate documents with DIFFERENT summaries: 0
identical summaries used for different train documents: 29 groups (58 docs)
ids shared between train and test: 0


## 4. Writing style and what summaries look like

Phase 4 surprise: train was cased, test lower-case. We check documents and summaries separately
(share of texts with < 1% upper-case letters). We also look at how summaries **start** (typical first words):
if most summaries begin the same way (e.g. "The appellant ..."), the model learns that quickly and it is free
BERTScore; and at sentence counts.


In [5]:
def style(t):
    t = t[:20000]
    letters = sum(c.isalpha() for c in t)
    upper = sum(c.isupper() for c in t)
    return "lower" if upper < 0.01 * max(letters, 1) else "cased"

st = {}
for split, docs in DATA.items():
    for field in ("document", "summary"):
        c = collections.Counter(style(d[field]) for d in docs if d[field])
        if c:
            st[f"{split}/{field}"] = {"cased": c.get("cased", 0), "lower": c.get("lower", 0),
                                      "lower %": round(100 * c.get("lower", 0) / sum(c.values()), 1)}
display(pd.DataFrame(st).T)

starts3 = collections.Counter(" ".join(su.clean_summ(d["summary"]).split()[:3]) for d in TR if d["summary"])
starts1 = collections.Counter(" ".join(su.clean_summ(d["summary"]).split()[:1]) for d in TR if d["summary"])
print("most common first word of a summary:", starts1.most_common(10))
print("most common first 3 words:")
display(pd.DataFrame(starts3.most_common(15), columns=["first 3 words", "summaries"]).assign(
    pct=lambda f: (100 * f["summaries"] / len(TR)).round(1)))
SENTS = {"document": [len(su.split_sentences(su.clean_summ(d["document"]))) for d in tqdm(TR[:2000], desc="sentences", leave=False)],
         "summary": [len(su.split_sentences(su.clean_summ(d["summary"]))) for d in TR[:2000] if d["summary"]]}
display(pd.DataFrame({k: pct_row(v) for k, v in SENTS.items()}).T.rename_axis("sentences (first 2,000 train)"))
for d in [d for d in TR if d["summary"]][1:3]:
    print("\n--- example id", d["id"], "| document words", len(d["document"].split()), "| summary words", len(d["summary"].split()))
    print("SUMMARY:", su.clean_summ(d["summary"])[:900], "...")

,cased,lower,lower %
train/document,7030.0,0.0,0.0
train/summary,7000.0,28.0,0.4
test/document,100.0,0.0,0.0


most common first word of a summary: [('The', 3845), ('In', 410), ('Section', 278), ('A', 262), ('On', 193), ('%', 180), ('Under', 175), ('By', 112), ('One', 79), ('An', 69)]
most common first 3 words:


,first 3 words,summaries,pct
0,The appellant was,319,4.5
1,The respondent was,112,1.6
2,The appellants were,87,1.2
3,"The appellant, a",83,1.2
4,The appellant company,67,1.0
5,The petitioner was,55,0.8
6,The appellant and,53,0.8
7,The appellant who,46,0.7
8,The respondents were,42,0.6
9,The first respondent,38,0.5


,mean,p10,p25,p50,p75,p90,p99,max
"sentences (first 2,000 train)",,,,,,,,
document,130.7,46.0,66.0,96.0,147.0,231.0,712.0,2617.0
summary,27.2,9.0,13.0,21.0,32.0,50.0,152.0,467.0



--- example id 2858 | document words 4853 | summary words 705
SUMMARY: The appellant executed a usufructuary mortgage of his house and continued to reside in it as a tenant under a lease obtained from the mortgagee. In 1949, the mortgagee left for Pakistan. He was declared an evacuee and his mortgagee interest in the mortgaged property vested in the Custodian under section 8 of the . Under section 12, the Custodian evicted the appellant and allotted it to others as tenants. In spite of demands by the appellant and the appellant tendering the mortgage amount, the Custodian refused to hand over vacant possession of the house to the appellant. The appellant applied to the Competent Officer under the . The Officer determined the mortgage debt due from the appellant and the appellant claimed vacant possession against tender of payment by him of the amount. The Officer rejected the claim and ordered the sale of the property under s.10(b)(ii) of the Separation Act ...

--- example id 6466 | 

## 5. Lengths in words and the compression ratio

**Compression ratio** = document words / summary words (10 = the summary is 10x shorter).
If summary length grows with document length, the model must learn a **length that depends on the document**
(a "generation length control" in the doc's list). We fit a simple rule on the train documents,
`log(summary words) = a + b * log(document words)`, i.e. `summary words = e^a * doc_words^b`, and check how good
it is (median error in %). A constant length is the alternative.


In [6]:
W = {split: {"document": np.array([len(d["document"].split()) for d in docs]),
             "summary": np.array([len(d["summary"].split()) for d in docs if d["summary"]])} for split, docs in DATA.items()}
rows = {f"{s}/{f}": pct_row(v) for s, x in W.items() for f, v in x.items() if len(v)}
display(pd.DataFrame(rows).T.rename_axis("words"))

# pairs only: documents whose summary is empty are left out of every document-vs-summary statistic
HAS_SUM = [d for d in TR if d["summary"] and d["summary"].strip()]
NO_SUM = [d["id"] for d in TR if not (d["summary"] and d["summary"].strip())]
print(f"train documents WITHOUT a summary: {len(NO_SUM)} {NO_SUM[:10]} -> left out of training and of the split")
dw = np.array([len(d["document"].split()) for d in HAS_SUM])
sw = np.array([len(d["summary"].split()) for d in HAS_SUM])
ratio = dw / np.maximum(sw, 1)
display(pd.DataFrame({"compression (doc/summary words)": pct_row(ratio)}).T)
print(f"Spearman(document words, summary words) = {spearman(dw, sw):.3f}")

ok = (dw > 0) & (sw > 0)
b, a = np.polyfit(np.log(dw[ok]), np.log(sw[ok]), 1)
pred_len = np.exp(a) * dw ** b
err_fit = np.median(np.abs(pred_len - sw) / sw)
err_const = np.median(np.abs(np.median(sw) - sw) / sw)
print(f"length rule: summary_words = {np.exp(a):.2f} * doc_words^{b:.3f}  -> median error {100 * err_fit:.0f}% "
      f"(constant {int(np.median(sw))} words: {100 * err_const:.0f}%)")
LEN_RULE = {"a": float(a), "b": float(b), "median_err": float(err_fit), "const_median_err": float(err_const),
            "const_words": int(np.median(sw))}

q = pd.qcut(dw, 5, duplicates="drop")
display(pd.DataFrame({"doc words": dw, "summary words": sw, "ratio": ratio}).groupby(q, observed=True).agg(
    docs=("summary words", "size"), summary_median=("summary words", "median"), ratio_median=("ratio", "median")
).rename_axis("document length quintile"))

,mean,p10,p25,p50,p75,p90,p99,max
words,,,,,,,,
train/document,4368.5,1348.0,2017.0,3128.0,4971.0,7851.0,24245.0,139645.0
train/summary,840.0,293.0,421.0,638.0,963.0,1501.0,4250.0,27768.0
test/document,4782.7,1503.0,2099.0,3171.0,5408.0,8562.0,29199.0,39695.0


train documents WITHOUT a summary: 2 ['299', '4799'] -> left out of training and of the split


,mean,p10,p25,p50,p75,p90,p99,max
compression (doc/summary words),6.0,2.0,3.0,4.0,7.0,10.0,25.0,100.0


Spearman(document words, summary words) = 0.616
length rule: summary_words = 4.62 * doc_words^0.612  -> median error 32% (constant 638 words: 40%)


,docs,summary_median,ratio_median
document length quintile,,,
"(92.999, 1794.4]",1406,395.0,2.992566
"(1794.4, 2640.8]",1405,549.0,3.984568
"(2640.8, 3713.2]",1406,644.0,4.811109
"(3713.2, 5607.6]",1405,788.0,5.688826
"(5607.6, 139645.0]",1406,1283.0,6.789190


## 6. Lengths in bpe32k tokens: what fits in my Phase-1 context?

My Phase-1 models read **1536 tokens**. For a **decoder-only** model (A2, A, B) the document AND the summary must
share these 1536 positions (`[document] [separator] [summary]`), so a long summary leaves little room for the
document. The **encoder-decoder** D reads the document in the encoder (it saw at most 1024 tokens in Phase 1) and
writes the summary in the decoder.

Tables: tokens per document / summary, and for the decoder-only layout the **room left for the document** =
1536 - summary tokens - 1. Also a rough **time per epoch** (Phase-5 training speed ~26k tokens/s for a 10-layer
model) and **generation time for the test set** (Phase 1: ~2 s per 128 words with the KV cache).
Cleaning = Phase-1 cleaning, cased (`su.clean_summ`).


In [7]:
from tokenizers import Tokenizer
tok = Tokenizer.from_file(str(TOK_PATH))
print("tokenizer:", TOK_PATH.relative_to(ROOT), "| vocab", tok.get_vocab_size())

def count_tokens(texts, desc):
    out = []
    for i in tqdm(range(0, len(texts), 256), desc=desc):
        out += [len(e.ids) for e in tok.encode_batch([" " + su.clean_summ(t) for t in texts[i:i + 256]], add_special_tokens=False)]
    return out

TOKS = load_cache("token_counts.json")
if TOKS is not None and len(TOKS["train_doc"]) == len(TR) and len(TOKS["test_doc"]) == len(TE):
    print("loaded cached token counts")
else:
    TOKS = {"train_doc": count_tokens([d["document"] for d in TR], "train documents"),
            "train_sum": count_tokens([d["summary"] or "" for d in TR], "train summaries"),
            "test_doc": count_tokens([d["document"] for d in TE], "test documents")}
    save_cache("token_counts.json", TOKS)
T = {k: np.array(v) for k, v in TOKS.items()}
pair = np.array([bool(d["summary"] and d["summary"].strip()) for d in TR])     # same pairs as section 5
T["train_doc"], T["train_sum"] = T["train_doc"][pair], T["train_sum"][pair]
display(pd.DataFrame({k: pct_row(v) for k, v in T.items()}).T.rename_axis("bpe32k tokens"))
print(f"tokens per word: documents {T['train_doc'].sum() / max(dw.sum(), 1):.3f}, summaries {T['train_sum'].sum() / max(sw.sum(), 1):.3f}")

CTX = 1536
room = CTX - T["train_sum"] - 1
fit = {f"summary <= {L} tokens %": round(100 * float(np.mean(T["train_sum"] <= L)), 1) for L in (256, 384, 512, 768, 1024, 1535)}
fit.update({f"document <= {L} tokens % (train / test)": f"{100 * np.mean(T['train_doc'] <= L):.1f} / {100 * np.mean(T['test_doc'] <= L):.1f}"
            for L in (512, 1024, 1536, 3072, 6144)})
fit["decoder-only: document + summary fit in 1536 %"] = round(100 * float(np.mean(T["train_doc"] + T["train_sum"] + 1 <= CTX)), 1)
fit["decoder-only: summary alone does not fit %"] = round(100 * float(np.mean(room <= 0)), 1)
display(pd.Series(fit).to_frame("share"))
display(pd.DataFrame({"room left for the document (decoder-only, 1536)": pct_row(np.clip(room, 0, None))}).T)

TRAIN_TPS, GEN_WPS = 26000, 64            # Phase-5 training tokens/s; Phase-1 generation ~128 words per 2 s
est = {}
for inp in (512, 1024, 1536):
    toks = np.minimum(T["train_doc"], inp).sum() + T["train_sum"].sum()
    est[f"input {inp} + full summary"] = {"tokens/epoch (M)": round(toks / 1e6, 1), "min/epoch (est.)": round(toks / TRAIN_TPS / 60, 1)}
display(pd.DataFrame(est).T)
test_pred_words = np.exp(a) * W["test"]["document"] ** b
print(f"generation estimate for the test set ({len(TE)} docs, predicted length {np.median(test_pred_words):.0f} words median): "
      f"~{test_pred_words.sum() / GEN_WPS / 60:.0f} min per model and decoding setting")

tokenizer: data\tokenizer\bpe32k.json | vocab 32000


test documents: 100%|██████████| 1/1 [00:00<00:00,  2.27it/s]


,mean,p10,p25,p50,p75,p90,p99,max
bpe32k tokens,,,,,,,,
train_doc,5079.5,1578.0,2347.0,3656.0,5770.0,9091.0,28078.0,161144.0
train_sum,992.8,342.0,495.0,750.0,1140.0,1772.0,5060.0,32347.0
test_doc,5566.0,1765.0,2440.0,3769.0,6366.0,10311.0,33697.0,45533.0


tokens per word: documents 1.163, summaries 1.182


,share
summary <= 256 tokens %,4.1
summary <= 384 tokens %,14.0
summary <= 512 tokens %,26.8
summary <= 768 tokens %,51.7
summary <= 1024 tokens %,69.6
summary <= 1535 tokens %,86.2
document <= 512 tokens % (train / test),0.5 / 0.0
document <= 1024 tokens % (train / test),3.2 / 4.0
document <= 1536 tokens % (train / test),9.3 / 6.0
document <= 3072 tokens % (train / test),40.1 / 39.0


,mean,p10,p25,p50,p75,p90,p99,max
"room left for the document (decoder-only, 1536)",703.0,0.0,394.0,785.0,1040.0,1193.0,1376.0,1492.0


,tokens/epoch (M),min/epoch (est.)
input 512 + full summary,10.6,6.8
input 1024 + full summary,14.1,9.0
input 1536 + full summary,17.5,11.2


generation estimate for the test set (100 docs, predicted length 644 words median): ~20 min per model and decoding setting


## 7. What the metric actually sees: BERTScore reads at most 512 roberta tokens

BERTScore (course evaluator: `bert_score.score(..., lang="en")` = **roberta-large**, layer 17) turns each summary
into roberta tokens and **cuts everything after 512 tokens** (510 + 2 special tokens). So if the reference
summaries are long, only their **beginning** counts, and anything our model writes after ~510 roberta tokens
is invisible to BERTScore (but still counts for ROUGE). This decides how long our summaries should be.

We count roberta-large tokens of the train summaries (tokenizer only, from the local Hugging Face cache that
bert_score already used in Phase 1; skipped if it is not available).


In [8]:
RB = load_cache("roberta_counts.json")
if RB is None:
    try:
        from transformers import AutoTokenizer
        try:
            rtok = AutoTokenizer.from_pretrained("roberta-large", local_files_only=True)
        except Exception:
            rtok = AutoTokenizer.from_pretrained("roberta-large")
        sums = [su.clean_summ(d["summary"]) for d in TR if d["summary"]]
        n_rb = []
        for i in tqdm(range(0, len(sums), 256), desc="roberta tokens"):
            n_rb += [len(x) for x in rtok(sums[i:i + 256], add_special_tokens=False)["input_ids"]]
        # word position where the 510-token cut happens (for the summaries that are cut)
        cut_words = []
        for s, n in zip(sums, n_rb):
            if n > 510:
                ids = rtok(s, add_special_tokens=False)["input_ids"][:510]
                cut_words.append(len(rtok.decode(ids).split()))
        RB = {"n": n_rb, "cut_words": cut_words}
        save_cache("roberta_counts.json", RB)
    except Exception as e:
        print("roberta-large tokenizer not available -> section skipped:", repr(e)[:200])
if RB is not None:
    n_rb = np.array(RB["n"])
    print(f"roberta tokens per train summary: {pct_row(n_rb)}")
    print(f"roberta tokens per word: {n_rb.sum() / max(sw.sum(), 1):.3f}")
    print(f"reference summaries LONGER than BERTScore's 510-token limit: {100 * np.mean(n_rb > 510):.1f}%")
    if RB["cut_words"]:
        print(f"  for those, BERTScore only sees the first {int(np.median(RB['cut_words']))} words (median; "
              f"p10 {int(np.percentile(RB['cut_words'], 10))}, p90 {int(np.percentile(RB['cut_words'], 90))})")

roberta tokens: 100%|██████████| 28/28 [00:01<00:00, 16.69it/s]


roberta tokens per train summary: {'mean': 1027.1, 'p10': 354, 'p25': 513, 'p50': 777, 'p75': 1179, 'p90': 1842, 'p99': 5232, 'max': 33427}
roberta tokens per word: 1.223
reference summaries LONGER than BERTScore's 510-token limit: 75.4%
  for those, BERTScore only sees the first 426 words (median; p10 396, p90 450)


## 8. How abstractive are the summaries?

- **novel n-grams** = share of the summary's word n-grams that never appear in the document
  (unigram novel 5% = almost every word is in the document; 4-gram novel 80% = sentences are re-phrased).
- **extractive fragments** (Grusky et al. 2018): greedily cover the summary with pieces copied word-for-word from
  the document. _coverage_ = share of summary words inside a copied piece; _density_ = average squared piece
  length (high = long copied passages, i.e. close to extractive).
- **copy availability** in my own tokens: share of summary bpe32k tokens that occur somewhere in the document
  (Phase 1: 68% of next tokens were already in the context -> that is why the copy mechanism helped).
  Here the copy source is the document, so this is the upper limit of what A2 / D can copy.

Words are lower-cased, punctuation dropped (only for these statistics). Sample of N_SAMPLE train documents.


In [9]:
rng = np.random.default_rng(0)
SAMPLE = [HAS_SUM[i] for i in sorted(rng.choice(len(HAS_SUM), size=min(N_SAMPLE, len(HAS_SUM)), replace=False))]
AB = load_cache("abstractiveness.json")
if AB is None:
    rows = []
    for d in tqdm(SAMPLE, desc="abstractiveness"):
        doc_c, sum_c = su.clean_summ(d["document"]), su.clean_summ(d["summary"])
        D, S = su.norm_words(doc_c), su.norm_words(sum_c)
        cov, dens, fr = su.fragments(S, D)
        dt = set(tok.encode(" " + doc_c, add_special_tokens=False).ids)
        st_ids = tok.encode(" " + sum_c, add_special_tokens=False).ids
        rows.append({"id": d["id"], **{f"novel {n}-gram %": 100 * su.novel_ngram_share(S, D, n) for n in (1, 2, 3, 4)},
                     "fragment coverage %": 100 * cov, "fragment density": dens,
                     "longest copied piece (words)": max(fr) if fr else 0,
                     "summary tokens in document %": 100 * np.mean([t in dt for t in st_ids]) if st_ids else np.nan})
    AB = rows
    save_cache("abstractiveness.json", AB)
ab = pd.DataFrame(AB).set_index("id")
display(ab.describe(percentiles=[0.1, 0.5, 0.9]).T[["mean", "10%", "50%", "90%"]].round(2))

abstractiveness: 100%|██████████| 500/500 [00:09<00:00, 54.15it/s]


,mean,10%,50%,90%
novel 1-gram %,10.03,5.73,9.31,14.56
novel 2-gram %,27.80,17.81,26.83,39.93
novel 3-gram %,40.88,25.86,38.92,59.03
novel 4-gram %,49.98,32.12,48.30,70.32
fragment coverage %,95.66,93.21,96.20,97.98
fragment density,15.36,4.91,12.59,28.56
longest copied piece (words),51.19,17.00,43.00,99.00
summary tokens in document %,93.69,90.40,94.25,96.75


## 9. Where in the document does the summary content come from?

1. **Sentence alignment:** every summary sentence is matched to the document sentence with the highest word-overlap
   F1 (content words: lower-case, >= 3 letters, no stop words). We record the matched sentence's **relative position**
   in the document (0 = first sentence, 1 = last) -> histogram in 10 bins. If most matches are near 0, the start of
   the document is the important part ("lead bias"); a flat histogram means the whole document matters.
2. **Order:** Spearman correlation between the summary sentence index and its matched position. Close to 1 = the
   summary follows the document order -> summarising chunks one after another and joining them is reasonable.
3. **Coverage by input window:** share of the summary's word **bigrams** that occur inside the part of the document a
   model could read (first 512 / 768 / 1024 / 1536 tokens, last 768, first 768 + last 768, whole). Windows are given in
   bpe32k tokens and converted to words with each document's own tokens/word ratio. This is the summarisation version
   of Phase 4/5's "which part of the case carries the signal".


In [10]:
STOP = set("the of and to in a is that by for on was be as with which it this an or from at are not has have had were "
           "been his her their its he she they them who whom any all such said shall may also under upon there than "
           "other into would could should will can these those being only per but out off our your one two".split())

def content(s):
    return {w for w in su.norm_words(s) if len(w) >= 3 and w not in STOP}

def f1_overlap(a, b):
    ov = len(a & b)
    return 0.0 if not ov else 2 * ov / (len(a) + len(b))

def bigrams(ws):
    return set(zip(ws, ws[1:]))

POS = load_cache("positions.json")
if POS is None:
    POS = {"rel": [], "order_rho": [], "coverage": []}
    idx = {d["id"]: i for i, d in enumerate(TR)}
    for d in tqdm(SAMPLE, desc="alignment"):
        ntok = TOKS["train_doc"][idx[d["id"]]]
        doc_c, sum_c = su.clean_summ(d["document"]), su.clean_summ(d["summary"])
        ds, ss = su.split_sentences(doc_c), su.split_sentences(sum_c)
        if len(ds) < 2 or not ss:
            continue
        dsets = [content(s) for s in ds]
        matched = []
        for s in ss:
            cs = content(s)
            if len(cs) < 3:
                continue
            j = int(np.argmax([f1_overlap(cs, x) for x in dsets]))
            matched.append(j / (len(ds) - 1))
        POS["rel"] += matched
        if len(matched) >= 3:
            POS["order_rho"].append(spearman(np.arange(len(matched)), matched))
        Dw, Sw = su.norm_words(doc_c), su.norm_words(sum_c)
        tpw = max(ntok, 1) / max(len(doc_c.split()), 1)          # tokens per word of THIS document
        w = lambda n_tok: int(round(n_tok / tpw))
        sb = bigrams(Sw)
        if not sb:
            continue
        windows = {"first 512": Dw[:w(512)], "first 768": Dw[:w(768)], "first 1024": Dw[:w(1024)],
                   "first 1536": Dw[:w(1536)], "last 768": Dw[-w(768):],
                   "first 768 + last 768": Dw[:w(768)] + ["<cut>"] + Dw[-w(768):], "whole document": Dw}
        POS["coverage"].append({k: len(sb & bigrams(v)) / len(sb) for k, v in windows.items()} | {"doc_tokens": int(ntok)})
    save_cache("positions.json", POS)

rel = np.array(POS["rel"])
hist = np.histogram(rel, bins=10, range=(0, 1))[0]
display(pd.DataFrame({"share of summary sentences %": (100 * hist / max(hist.sum(), 1)).round(1)},
                     index=[f"{i / 10:.1f}-{(i + 1) / 10:.1f}" for i in range(10)]).T.rename_axis("relative position in document"))
print(f"summary order vs document order: median Spearman {np.nanmedian(POS['order_rho']):.2f} "
      f"(share of summaries with rho > 0.5: {100 * np.nanmean(np.array(POS['order_rho']) > 0.5):.0f}%)")
cov = pd.DataFrame(POS["coverage"])
long_ = cov["doc_tokens"] > 1536
tbl = pd.DataFrame({"all sampled docs": cov.drop(columns="doc_tokens").mean(),
                    "docs > 1536 tokens": cov[long_].drop(columns="doc_tokens").mean()}).mul(100).round(1)
print("share of summary bigrams found in the part of the document a model reads (%):")
display(tbl)

alignment: 100%|██████████| 500/500 [00:04<00:00, 109.97it/s]


,0.0-0.1,0.1-0.2,0.2-0.3,0.3-0.4,0.4-0.5,0.5-0.6,0.6-0.7,0.7-0.8,0.8-0.9,0.9-1.0
relative position in document,,,,,,,,,,
share of summary sentences %,6.6,10.3,10.3,8.6,9.1,10.1,9.9,10.5,12.5,12.1


summary order vs document order: median Spearman 0.73 (share of summaries with rho > 0.5: 79%)
share of summary bigrams found in the part of the document a model reads (%):


,all sampled docs,docs > 1536 tokens
first 512,17.9,16.5
first 768,24.7,22.8
first 1024,30.7,28.2
first 1536,40.3,37.9
last 768,30.3,28.1
first 768 + last 768,44.7,42.7
whole document,72.2,72.8


## 10. Validation split (no public dev set)

The doc (12.3.2) lets us build our own held-out split from train. Rule (same idea as Phase 1's 1% split):
`md5(document text) % 1000 < 50` -> **~5% validation**. It does not depend on the file order, and identical
documents always land on the same side (no leak through duplicates). Inside validation, a fixed **local-eval set**
of 150 documents is used for the slow part (generate summaries + BERTScore); the whole validation set is used for
the cheap validation loss during training. Saved to `data/processed/phase6/split_v1.json` (loaded on re-runs, so it
never changes silently). Test documents that are also in train: listed here (they could be used as a sanity check).


In [11]:
SPLIT = su.make_val_split(HAS_SUM, val_per_mille=50, n_local=150, path=SPLIT_PATH)   # no empty summaries
SPLIT.setdefault("dropped_no_summary", NO_SUM)
by_id = {d["id"]: d for d in TR}
vs = set(SPLIT["val"])
print(f"train {len(SPLIT['train']):,} | val {len(SPLIT['val']):,} | local_eval {len(SPLIT['local_eval'])} | rule: {SPLIT['rule']}")
cmp = {name: {"docs": len(ids),
              "doc words median": int(np.median([len(by_id[i]["document"].split()) for i in ids])),
              "summary words median": int(np.median([len(by_id[i]["summary"].split()) for i in ids]))}
       for name, ids in (("train part", SPLIT["train"]), ("val", SPLIT["val"]), ("local_eval", SPLIT["local_eval"]))}
display(pd.DataFrame(cmp).T)

train 6,661 | val 367 | local_eval 150 | rule: md5(document) % 1000 < 50; local_eval = 150 smallest fingerprints


,docs,doc words median,summary words median
train part,6661,3129,638
val,367,3125,650
local_eval,150,3360,582


## 11. Baselines with the real metric (local-eval set, 150 documents)

No model yet; these show the **floor** and how the metric reacts to **length**:

| baseline                         | what it outputs                                                                                                                                  |
| -------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------ |
| same summary for every case      | one fixed train summary (median length) -> how high BERTScore is for unrelated legal text                                                        |
| lead-k                           | the first k words of the document (k = 64 ... 768, and the median summary length)                                                                |
| lead, predicted length           | first k words with k from the length rule of section 5                                                                                           |
| tail                             | the last (median length) words                                                                                                                   |
| head + tail                      | first half + last half of the median length                                                                                                      |
| random sentences                 | random document sentences in document order up to the predicted length                                                                           |
| **oracle extractive**            | greedily chosen document sentences that maximise ROUGE-1 with the **reference** (uses the answer -> upper bound for any sentence-picking method) |
| reference cut to 128 / 256 words | uses the answer: shows how much BERTScore drops only because of length                                                                           |

BERTScore runs on the GPU (roberta-large, same settings as the course evaluator). ROUGE = whitespace tokens,
case-sensitive (like the course's Phase-1 evaluator). Cached in `baselines.json`.


In [12]:
LE = [by_id[i] for i in SPLIT["local_eval"]]
refs = [su.clean_summ(d["summary"]) for d in LE]
docs_c = [su.clean_summ(d["document"]) for d in LE]
med = LEN_RULE["const_words"]
train_part = [by_id[i] for i in SPLIT["train"]]
fixed = sorted(train_part, key=lambda d: abs(len(d["summary"].split()) - med))[0]
fixed_sum = su.clean_summ(fixed["summary"])

def first_words(t, k): return " ".join(t.split()[:max(1, int(k))])
def last_words(t, k): return " ".join(t.split()[-max(1, int(k)):])
def pred_k(t): return int(np.clip(np.exp(LEN_RULE["a"]) * len(t.split()) ** LEN_RULE["b"], 32, 2000))

def oracle(doc, ref, max_ratio=1.5):
    # greedy: add the sentence that raises ROUGE-1 F1 (lower-case words) most; stop when nothing helps
    sents = su.split_sentences(doc)
    sc = [collections.Counter(su.norm_words(s)) for s in sents]
    rc = collections.Counter(su.norm_words(ref)); rn = sum(rc.values())
    chosen, sel, sel_n, ov = [], collections.Counter(), 0, 0
    best_f = 0.0
    while sel_n < max_ratio * rn:
        best = None
        for j, c in enumerate(sc):
            if j in chosen or not c:
                continue
            gain = sum(min(sel[w] + n, rc[w]) - min(sel[w], rc[w]) for w, n in c.items() if w in rc)
            o, n_ = ov + gain, sel_n + sum(c.values())
            f = 2 * o / (n_ + rn) if o else 0.0
            if f > best_f + 1e-9:
                best_f, best = f, (j, o, n_)
        if best is None:
            break
        j, ov, sel_n = best
        chosen.append(j); sel.update(sc[j])
    return " ".join(sents[j] for j in sorted(chosen)) or first_words(doc, med)

def random_sents(doc, k, seed):
    sents = su.split_sentences(doc)
    r = np.random.default_rng(seed); order = r.permutation(len(sents)); pick, n = [], 0
    for j in order:
        if n >= k: break
        pick.append(j); n += len(sents[j].split())
    return " ".join(sents[j] for j in sorted(pick))

CANDS = {"same summary for every case": [fixed_sum] * len(LE)}
for k in sorted({64, 128, 256, 384, 512, 768, med}):
    CANDS[f"lead-{k}"] = [first_words(t, k) for t in docs_c]
CANDS["lead, predicted length"] = [first_words(t, pred_k(t)) for t in docs_c]
CANDS[f"tail-{med}"] = [last_words(t, med) for t in docs_c]
CANDS[f"head {med // 2} + tail {med // 2}"] = [first_words(t, med // 2) + " " + last_words(t, med // 2) for t in docs_c]
CANDS["random sentences, predicted length"] = [random_sents(t, pred_k(t), i) for i, t in enumerate(docs_c)]
CANDS["ORACLE extractive (uses reference)"] = [oracle(t, r) for t, r in tqdm(zip(docs_c, refs), total=len(LE), desc="oracle")]
CANDS["reference cut to 128 words (uses reference)"] = [first_words(r, 128) for r in refs]
CANDS["reference cut to 256 words (uses reference)"] = [first_words(r, 256) for r in refs]

BASE = load_cache("baselines.json") or {}
for name, preds in tqdm(CANDS.items(), desc="scoring baselines"):
    if name not in BASE:
        BASE[name] = su.score_summaries(preds, refs)
        save_cache("baselines.json", BASE)
base_tbl = pd.DataFrame(BASE).T[["pred_words", "bertscore_f1", "bertscore_p", "bertscore_r", "rouge1_f1", "rouge2_f1", "rougeL_f1"]]
display(base_tbl.round(4))
print(f"reference summaries in local_eval: median {int(np.median([len(r.split()) for r in refs]))} words")

Loading weights: 100%|██████████| 389/389 [00:01<00:00, 320.16it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:01<00:00, 248.83it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPE

,pred_words,bertscore_f1,bertscore_p,bertscore_r,rouge1_f1,rouge2_f1,rougeL_f1
same summary for every case,638.0000,0.8043,0.8024,0.8063,0.3173,0.0683,0.1576
lead-64,64.0000,0.7929,0.8084,0.7782,0.0764,0.0186,0.0583
lead-128,128.0000,0.8058,0.8133,0.7985,0.1641,0.0559,0.1017
lead-256,255.5533,0.8239,0.8223,0.8255,0.2946,0.1174,0.1562
lead-384,382.7000,0.8314,0.8245,0.8386,0.3726,0.1627,0.1894
lead-512,509.8467,0.8317,0.8242,0.8394,0.4147,0.1871,0.2084
lead-638,635.0067,0.8317,0.8242,0.8394,0.4373,0.2057,0.2190
lead-768,763.6333,0.8317,0.8242,0.8394,0.4461,0.2191,0.2281
"lead, predicted length",769.4800,0.8313,0.8242,0.8386,0.4574,0.2089,0.2226
tail-638,635.0067,0.8344,0.8323,0.8366,0.4721,0.2473,0.2493


reference summaries in local_eval: median 582 words


## 12. Save the summary


In [13]:
summary = {
    "format": INFO, "duplicates": DUP_INFO,
    "words": {f"{s}/{f}": pct_row(v) for s, x in W.items() for f, v in x.items() if len(v)},
    "compression": pct_row(ratio), "spearman_doc_sum_words": spearman(dw, sw), "length_rule": LEN_RULE,
    "tokens": {k: pct_row(v) for k, v in T.items()}, "context_fit": fit,
    "roberta": None if RB is None else {"tokens": pct_row(RB["n"]), "share_over_510": float(np.mean(np.array(RB["n"]) > 510)),
                                        "cut_words_median": int(np.median(RB["cut_words"])) if RB["cut_words"] else None},
    "abstractiveness_mean": ab.mean().round(3).to_dict(),
    "position_hist_pct": (100 * hist / max(hist.sum(), 1)).round(1).tolist(),
    "order_rho_median": float(np.nanmedian(POS["order_rho"])) if POS["order_rho"] else None,
    "bigram_coverage_pct": tbl.to_dict(),
    "split": {k: len(v) for k, v in SPLIT.items() if isinstance(v, list)},
    "baselines": BASE,
}
save_cache("data_summary.json", summary)
print("saved", (OUT / "data_summary.json").relative_to(ROOT))

saved experiments\phase6\p6_01_data\data_summary.json


## Findings (written after the run, see `docs/PROJECT_NOTES.md` section 13.2)
